In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 MB 94.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import mlflow
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TARGET_COLUMN = "Churn_Value"

# Load feature table and raw table
df = spark.table("telco_churn_features").toPandas()
raw_df = spark.table("telco_customer_churn").toPandas()

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Load registered model
model = mlflow.sklearn.load_model(
    "models:/workspace.default.churn-calibrated-model/1"
)

print(f"Model loaded: {type(model)}")
print(f"Scoring {len(X)} customers")

Model loaded: <class 'sklearn.calibration.CalibratedClassifierCV'>
Scoring 7043 customers


In [0]:
# Config values — exact from business.yaml
CONTACT_COST   = 5
DISCOUNT       = 50
RESCUE_RATE    = 0.2
HORIZON_MONTHS = 12

# Generate calibrated probabilities
churn_probs = model.predict_proba(X)[:, 1]

# Build base table — mirrors build_policy_table()
monthly_charges = pd.to_numeric(
    raw_df["Monthly Charges"], errors="coerce"
).values

pred_df = pd.DataFrame({
    "CustomerID":        raw_df["CustomerID"].values,
    "churn_probability": churn_probs,
    "MonthlyCharges":    monthly_charges,
    "Churn Value":       y.values,
})

pred_df["ValueProxy"] = pred_df["MonthlyCharges"] * HORIZON_MONTHS

# Apply policy scores — mirrors apply_policy_scores()
def sensitivity_weight(p):
    return max(0.0, 1 - abs(p - 0.5) * 2)

pred_df["score_risk_only"] = pred_df["churn_probability"]

pred_df["score_risk_value"] = (
    pred_df["churn_probability"] * pred_df["ValueProxy"]
)

pred_df["weight"] = pred_df["churn_probability"].apply(sensitivity_weight)
pred_df["score_risk_value_weighted"] = (
    pred_df["churn_probability"] *
    pred_df["ValueProxy"] *
    pred_df["weight"]
)

print(f"Customers scored: {len(pred_df)}")
print(f"Avg churn probability: {churn_probs.mean():.3f}")
print(f"Avg monthly charges: ${monthly_charges.mean():.2f}")
print(pred_df[[
    "CustomerID", "churn_probability",
    "MonthlyCharges", "ValueProxy", "score_risk_value"
]].head(5).to_string())

Customers scored: 7043
Avg churn probability: 0.264
Avg monthly charges: $64.76
   CustomerID  churn_probability  MonthlyCharges  ValueProxy  score_risk_value
0  3668-QPYBK           0.023635           53.85       646.2         15.273241
1  9237-HQITU           0.026190           70.70       848.4         22.219441
2  9305-CDSKC           0.043164           99.65      1195.8         51.615131
3  7892-POOKP           0.454709          104.80      1257.6        571.841798
4  0280-XJGEX           0.002247          103.70      1244.4          2.796405


In [0]:
def simulate_policy(df, score_col, budget_pct,
                    contact_cost=CONTACT_COST,
                    discount=DISCOUNT,
                    rescue_rate=RESCUE_RATE):
    df = df.copy().sort_values(score_col, ascending=False)
    cutoff = max(1, int(len(df) * budget_pct))
    targeted = df.head(cutoff).copy()

    expected_retained_value = (
        targeted["churn_probability"] *
        targeted["ValueProxy"] *
        rescue_rate
    ).sum()

    realized_retained_value_proxy = (
        targeted["Churn Value"] *
        targeted["ValueProxy"] *
        rescue_rate
    ).sum()

    total_cost = cutoff * (contact_cost + discount)
    expected_profit = expected_retained_value - total_cost
    realized_profit_proxy = realized_retained_value_proxy - total_cost
    roi = expected_profit / total_cost if total_cost > 0 else 0

    return {
        "policy":                        score_col,
        "budget_pct":                    budget_pct,
        "customers_targeted":            cutoff,
        "rescue_rate":                   rescue_rate,
        "targeted_churn_rate":           targeted["Churn Value"].mean(),
        "avg_targeted_value":            targeted["ValueProxy"].mean(),
        "expected_retained_value":       expected_retained_value,
        "realized_retained_value_proxy": realized_retained_value_proxy,
        "total_cost":                    total_cost,
        "expected_profit":               expected_profit,
        "roi":                           roi,
        "realized_profit_proxy":         realized_profit_proxy,
    }

# Full grid — mirrors run_policy_comparison()
policies     = ["score_risk_only", "score_risk_value", "score_risk_value_weighted"]
budgets      = [0.05, 0.10, 0.15]
rescue_rates = [0.1, 0.2, 0.3]

results = []
for policy in policies:
    for b in budgets:
        for r in rescue_rates:
            result = simulate_policy(
                df=pred_df,
                score_col=policy,
                budget_pct=b,
                rescue_rate=r
            )
            results.append(result)

results_df = pd.DataFrame(results)

# Summary table
summary = (
    results_df.groupby(["policy", "budget_pct"])
    .agg({"expected_profit": "mean", "roi": "mean"})
    .reset_index()
)

print("\nPolicy Comparison Summary")
print(summary.to_string())

# Best policy
best = results_df.sort_values("expected_profit", ascending=False).iloc[0]
print(f"\nBest Policy: {best['policy']}")
print(f"Budget:      {best['budget_pct']}")
print(f"Rescue Rate: {best['rescue_rate']}")
print(f"Expected Profit: ${best['expected_profit']:,.0f}")
print(f"ROI: {best['roi']:.3f}")
print(f"Customers Targeted: {best['customers_targeted']}")


Policy Comparison Summary
                      policy  budget_pct  expected_profit       roi
0            score_risk_only        0.05     26494.017505  1.368493
1            score_risk_only        0.10     45758.642588  1.181783
2            score_risk_only        0.15     56707.804571  0.976374
3           score_risk_value        0.05     42697.458415  2.205447
4           score_risk_value        0.10     66263.492124  1.711351
5           score_risk_value        0.15     81780.221090  1.408062
6  score_risk_value_weighted        0.05     26874.413974  1.388141
7  score_risk_value_weighted        0.10     47663.651489  1.230983
8  score_risk_value_weighted        0.15     64125.912209  1.104096

Best Policy: score_risk_value
Budget:      0.15
Rescue Rate: 0.3
Expected Profit: $151,710
ROI: 2.612
Customers Targeted: 1056


In [0]:
def apply_score(df, policy):
    df = df.copy()
    if policy == "score_risk_only":
        df["score"] = df["churn_probability"]
    elif policy == "score_risk_value":
        df["score"] = df["churn_probability"] * df["ValueProxy"]
    elif policy == "score_risk_value_weighted":
        df["weight"] = df["churn_probability"].apply(sensitivity_weight)
        df["score"] = df["churn_probability"] * df["ValueProxy"] * df["weight"]
    return df

def find_optimal_threshold(df, budget_pct, policy,
                           contact_cost=CONTACT_COST,
                           discount=DISCOUNT,
                           rescue_rate=RESCUE_RATE):
    df = apply_score(df, policy)
    df = df.sort_values("score", ascending=False).reset_index(drop=True)

    n = max(1, int(len(df) * budget_pct))
    targeted = df.head(n).copy()

    expected_saved_value = (
        targeted["churn_probability"] *
        targeted["ValueProxy"] *
        rescue_rate
    ).sum()

    campaign_cost = n * (contact_cost + discount)
    expected_profit = expected_saved_value - campaign_cost
    roi = expected_profit / campaign_cost if campaign_cost > 0 else 0.0
    score_threshold = targeted["score"].iloc[-1]

    return {
        "budget_pct":                    float(budget_pct),
        "customers_targeted":            n,
        "score_threshold":               float(score_threshold),
        "expected_saved_value":          float(expected_saved_value),
        "campaign_cost":                 float(campaign_cost),
        "expected_profit":               float(expected_profit),
        "expected_profit_per_customer":  float(expected_profit / n) if n > 0 else 0.0,
        "roi":                           float(roi),
        "policy":                        policy,
    }, targeted

optimal, targeted_df = find_optimal_threshold(
    df=pred_df,
    budget_pct=best["budget_pct"],
    policy=best["policy"],
    rescue_rate=best["rescue_rate"]
)

print("Optimal Threshold Results:")
for k, v in optimal.items():
    print(f"  {k}: {v}")

Optimal Threshold Results:
  budget_pct: 0.15
  customers_targeted: 1056
  score_threshold: 444.55793837070456
  expected_saved_value: 209790.3316355404
  campaign_cost: 58080.0
  expected_profit: 151710.3316355404
  expected_profit_per_customer: 143.66508677607993
  roi: 2.612092486837817
  policy: score_risk_value


In [0]:
mlflow.set_experiment("/churn-retention-system")

with mlflow.start_run(run_name="xgboost_isotonic_full_pipeline"):

    # Ranking metrics
    def recall_at_k(y_true, probs, k):
        d = pd.DataFrame({"y": y_true, "p": probs})
        d = d.sort_values("p", ascending=False)
        cut = max(1, int(len(d) * k))
        return d.head(cut)["y"].sum() / d["y"].sum()

    def precision_at_k(y_true, probs, k):
        d = pd.DataFrame({"y": y_true, "p": probs})
        d = d.sort_values("p", ascending=False)
        cut = max(1, int(len(d) * k))
        return d.head(cut)["y"].mean()

    def lift_at_k(y_true, probs, k):
        d = pd.DataFrame({"y": y_true, "p": probs})
        d = d.sort_values("p", ascending=False)
        cut = max(1, int(len(d) * k))
        return d.head(cut)["y"].mean() / d["y"].mean()

    mlflow.log_param("model_type",   "xgboost")
    mlflow.log_param("calibration",  "isotonic")
    mlflow.log_param("best_policy",  best["policy"])
    mlflow.log_param("budget_pct",   best["budget_pct"])
    mlflow.log_param("rescue_rate",  best["rescue_rate"])
    mlflow.log_param("score_threshold", optimal["score_threshold"])

    mlflow.log_metric("precision_at_5",  precision_at_k(y.values, churn_probs, 0.05))
    mlflow.log_metric("precision_at_10", precision_at_k(y.values, churn_probs, 0.10))
    mlflow.log_metric("recall_at_5",     recall_at_k(y.values, churn_probs, 0.05))
    mlflow.log_metric("recall_at_10",    recall_at_k(y.values, churn_probs, 0.10))
    mlflow.log_metric("lift_at_10",      lift_at_k(y.values, churn_probs, 0.10))

    mlflow.log_metric("best_expected_profit",    best["expected_profit"])
    mlflow.log_metric("best_roi",                best["roi"])
    mlflow.log_metric("best_customers_targeted", best["customers_targeted"])
    mlflow.log_metric("optimal_threshold_profit", optimal["expected_profit"])
    mlflow.log_metric("optimal_threshold_roi",    optimal["roi"])

    # Nested policy runs — mirrors run_policy_comparison MLflow structure
    for _, row in results_df.iterrows():
        with mlflow.start_run(
            run_name=f"{row['policy']}_b{row['budget_pct']}_r{row['rescue_rate']}",
            nested=True
        ):
            mlflow.log_param("policy_type",  row["policy"])
            mlflow.log_param("budget_pct",   row["budget_pct"])
            mlflow.log_param("rescue_rate",  row["rescue_rate"])
            mlflow.log_param("contact_cost", CONTACT_COST)
            mlflow.log_param("discount",     DISCOUNT)
            mlflow.log_metric("expected_profit",             row["expected_profit"])
            mlflow.log_metric("roi",                         row["roi"])
            mlflow.log_metric("customers_targeted",          row["customers_targeted"])
            mlflow.log_metric("targeted_churn_rate",         row["targeted_churn_rate"])
            mlflow.log_metric("avg_targeted_value",          row["avg_targeted_value"])
            mlflow.log_metric("expected_retained_value",     row["expected_retained_value"])
            mlflow.log_metric("realized_profit_proxy",       row["realized_profit_proxy"])

    # Profit vs budget plot — mirrors plot_profit_vs_budget()
    plot_df = results_df[results_df["policy"] == best["policy"]].copy()
    plot_df = plot_df.groupby("budget_pct")["expected_profit"].mean().reset_index()
    optimal_row = plot_df.loc[plot_df["expected_profit"].idxmax()]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(plot_df["budget_pct"], plot_df["expected_profit"], marker="o")
    ax.scatter(optimal_row["budget_pct"], optimal_row["expected_profit"],
               color="red", s=120, label="Optimal")
    ax.axvline(optimal_row["budget_pct"], linestyle="--", color="red", alpha=0.6)
    ax.set_xlabel("Targeting Budget (%)")
    ax.set_ylabel("Expected Profit ($)")
    ax.set_title(f"Profit vs Budget — {best['policy']}")
    ax.legend()
    ax.grid(True)
    fig.savefig("/tmp/profit_vs_budget.png")
    mlflow.log_artifact("/tmp/profit_vs_budget.png")
    plt.close()

    # Artifacts
    results_df.to_csv("/tmp/policy_comparison.csv", index=False)
    mlflow.log_artifact("/tmp/policy_comparison.csv")
    targeted_df.to_csv("/tmp/targeted_customers.csv", index=False)
    mlflow.log_artifact("/tmp/targeted_customers.csv")

    print("Pipeline run logged to MLflow")
    print(f"Best policy: {best['policy']}")
    print(f"Expected profit: ${best['expected_profit']:,.0f}")
    print(f"ROI: {best['roi']:.3f}")

Pipeline run logged to MLflow
Best policy: score_risk_value
Expected profit: $151,710
ROI: 2.612


In [0]:
# Intervention decision table
intervention_table = pred_df[[
    "CustomerID", "churn_probability", "MonthlyCharges",
    "ValueProxy", "score_risk_only",
    "score_risk_value", "score_risk_value_weighted", "weight"
]].copy()

score_threshold = optimal["score_threshold"]
policy          = optimal["policy"]

intervention_table["intervene"] = (
    intervention_table[policy] >= score_threshold
).astype(int)

intervention_spark = spark.createDataFrame(intervention_table)
intervention_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("telco_churn_interventions")

print(f"Intervention table written: telco_churn_interventions")
print(f"Policy used: {policy}")
print(f"Score threshold: {score_threshold:.4f}")
print(f"Customers flagged: {intervention_table['intervene'].sum()}")
print(f"Expected monthly revenue protected: ${intervention_table[intervention_table['intervene']==1]['MonthlyCharges'].sum():,.0f}")

Intervention table written: telco_churn_interventions
Policy used: score_risk_value
Score threshold: 444.5579
Customers flagged: 1056
Expected monthly revenue protected: $93,888
